# Scratchpad (State 1)

Interactive derivations and numerical checks before promotion to the clean paper
(`docs/fund-flow-rotation.tex`). Results are marked `[VERIFIED]` only after an
independent numerical check and human consensus. Dead ends are kept, marked
`[DEAD-END]`.

## 2026-06-10 - Smoothing the rotation-graph coordinates

**Problem.** The rotation graph plots RS-Ratio $\mathrm{RS}_{c,t}$ (eq:rs_ratio)
against RS-Momentum $\mathrm{RS}^{\mathrm{m}}_{c,t}$ (eq:rs_momentum). Built on raw
monthly relative flow, the per-sector tails crisscross the plane instead of tracing
the clockwise arcs the graph is meant to show. Monthly $\mathrm{rel}_{c,t} = g_{c,t}
- g_{U,t}$ inherits the full noise of a single month of fund flow, so the tail is
dominated by high-frequency jitter rather than rotation.

**Proposed fix.** Smooth the relative-strength signal with a trailing $w$-month mean
before standardizing,
$$\bar{\mathrm{rel}}_{c,t}(w) = \frac{1}{w}\sum_{k=0}^{w-1}\mathrm{rel}_{c,t-k},$$
and define the rotation-graph relative strength as the standardization of the
smoothed series,
$$\mathrm{RS}_{c,t} = \frac{\bar{\mathrm{rel}}_{c,t} - \mu_L(\bar{\mathrm{rel}}_c)}
{\sigma_L(\bar{\mathrm{rel}}_c)},\qquad
\mathrm{RS}^{\mathrm{m}}_{c,t} = \mathrm{RS}_{c,t} - \mathrm{RS}_{c,t-D}.$$
This is the standard construction: the JdK / Bloomberg RRG is built on a smoothed
relative-strength line, not the raw ratio.

**Self-challenge.**
- `[CHECK]` Degenerate case $w=1$: $\bar{\mathrm{rel}} = \mathrm{rel}$, so the
  coordinates reduce exactly to the current unsmoothed definition. Good - smoothing
  is a strict generalization, the old behavior is $w=1$.
- `[CHECK]` Internal consistency: the mean is linear, so $\bar{\mathrm{rel}} =
  \overline{g_c} - \overline{g_U}$; smoothing the strength signal is the same as
  using smoothed growth against a smoothed baseline. No inconsistency.
- `[CHECK]` Over-smoothing: a large $w$ manufactures an artificially clean spiral
  and adds a lag of about $(w-1)/2$ months. Keep $w$ small; do not let the chart
  invent rotation that is not in the data.
- `[CHECK]` Cost: $\mathrm{RS}$ needs $w-1$ extra warm-up months before it is defined
  (the smoothing window must fill before the standardization window starts).
- `[CHECK]` Does it actually de-jitter? Needs an empirical measure, below.

**Numerical check.** Measure the mean per-sector tail path length in $(\mathrm{RS},
\mathrm{RS}^{\mathrm{m}})$ space over the last 6 months: the summed Euclidean length
of the polyline a sector traces. Genuine rotation has modest length; jitter inflates
it with back-and-forth zigzag. If smoothing helps, length should fall.

In [1]:
import numpy as np
import sanity_check as sc
from categories import category_panel
from rotation import relative_flow, z_score, momentum

panel = sc.load()
cat0 = relative_flow(category_panel(panel), panel)

def coords(w):
    c = cat0.sort_values(['category', 'month']).copy()
    c['rel_s'] = (c['rel'] if w == 1 else
                  c.groupby('category')['rel'].transform(
                      lambda s: s.rolling(w, min_periods=w).mean()))
    c = z_score(c, col='rel_s', lookback=12, out='rs')
    c = momentum(c, col='rs', lag=3, out='rs_mom')
    return c

def mean_tail_path(c, tail=6):
    months = sorted(c['month'].unique())[-(tail + 1):]
    sub = c[c['month'].isin(months)].dropna(subset=['rs', 'rs_mom'])
    L = []
    for _, d in sub.groupby('category'):
        d = d.sort_values('month')
        if len(d) < 2:
            continue
        dx = np.diff(d['rs'].values); dy = np.diff(d['rs_mom'].values)
        L.append(np.sqrt(dx**2 + dy**2).sum())
    return np.mean(L)

print('smoothing w | mean per-sector 6m tail path length in (rs,rs_mom)')
for w in [1, 2, 3, 4]:
    print(f'   w={w}   {mean_tail_path(coords(w)):.3f}')

smoothing w | mean per-sector 6m tail path length in (rs,rs_mom)
   w=1   17.112
   w=2   11.989
   w=3   11.459
   w=4   9.401


**Result.** Path length falls 17.1 -> 12.0 -> 11.5 -> 9.4 for $w = 1,2,3,4$. The
$w=1 \to 2$ step removes most of the jitter (a 30% drop); $w=2 \to 3$ adds little
(11.99 -> 11.46); $w=4$ keeps shrinking but at the cost of more lag, consistent with
the over-smoothing `[CHECK]`.

**Recommendation.** Adopt the trailing-mean smoothing with a default $w=3$: it
matches the quarterly cadence already used for the momentum lag $D=3$, and a quarter
is the natural unit for a flow-rotation read. $w=2$ is a defensible lighter
alternative that captures most of the de-jittering with one fewer month of lag.

Status: `[VERIFIED]`. Consensus 2026-06-10: adopt the trailing-mean smoothing with
default $w=3$. Promoted to State 2 - eq:rel_smoothed (smoothing) added and eq:rs_ratio
generalized to the smoothed signal.

## 2026-06-10 - Is net flow zero-sum across sectors?

Question (Max): would it be an interesting sanity check to see if net flow is
zero-sum? Two distinct senses of "zero-sum":

1. Raw dollar flow $\sum_c F_{c,t}$: no structural reason to vanish -- it is the
   common tide of money entering or leaving the sector-ETF complex as a whole.
   Its size relative to gross flow measures how much of sector flow is tide
   versus rotation.
2. Relative flow: the claim is that the $A_{t-1}$-weighted relative flows sum to
   zero each month, exactly, by construction.

Derivation of (2). From eq:relative_flow, eq:category_g, eq:universe_g, with the
universe being the same eleven categories:

$$\sum_c A_{c,t-1}\,\mathrm{rel}_{c,t}
  = \sum_c A_{c,t-1}\left(\frac{F_{c,t}}{A_{c,t-1}} - g_{U,t}\right)
  = \sum_c F_{c,t} - \frac{\sum_c F_{c,t}}{\sum_c A_{c,t-1}}\sum_c A_{c,t-1}
  = 0.$$

So relative flow is a dollar conservation law: every dollar of above-market flow
in one sector is matched by below-market flow elsewhere. Self-challenges:

- `[CHECK]` The identity needs $g_U$ computed over the same fund set with the
  same $A_{t-1}$ presence convention as the categories (funds entering
  mid-history). Both aggregations use the same skip-NaN sum, so entry months are
  consistent; verified numerically below, including such months.
  CORRECTION (State 0, same day): the consistency claim is wrong when a whole
  CATEGORY enters mid-history -- its first month has undefined growth, the
  category leaves the sum, and the residual is exactly $-F_{\mathrm{entrant},t}$
  (the unit test demonstrates the synthetic entry-month residual equals the
  entrant's flow to machine precision). A fund joining an existing multi-fund
  category stays consistent. Scope of the identity: months with no category
  entries; the live eleven-sector panel has none, so it holds in every month.
- `[CHECK]` Weighted, not unweighted: the unweighted sum of rel is materially
  nonzero (mean |sum| 0.077 pp/mo). Zero-sum holds for dollars, not for growth
  rates.
- `[CHECK]` Raw dollar flow is not zero-sum and should not be; empirically the
  common tide is large (below).
- `[CHECK]` First run of the check produced infs: it exposed an aggregation bug
  -- months where every member fund's AUM is NaN (2019-07/08, before the AUM
  anchoring point) were summed to a fabricated 0.0, making the next month's
  g = F/0 = inf. Fixed in categories.py (sum with min_count=1) with a
  regression test; Scenario B engineering fix, no formula change.

Empirical results (code below, live panel, 81 months):

- Invariant: max over months of $|\sum_c A_{c,t-1}\mathrm{rel}_{c,t}|$ is 1e-6
  dollars (relative 1.9e-16) -- machine precision. Holds.
- Raw flow: mean net +\$0.57B/month, +\$46.2B cumulative; median
  |net|/gross = 42%; 57% of months net positive. So on a typical month roughly
  40% of gross sector flow is common tide and 60% nets out as true rotation.

Status: `[VERIFIED]`. Consensus 2026-06-10: promote the identity to the paper's
Validation section (State 2, eq:flow_conservation), a live conservation check in
sanity_check.py (State 3), and an offline invariant unit test on synthetic data
(State 0).


In [1]:
import numpy as np
import pandas as pd
from viz import _load
from categories import category_panel
from rotation import relative_flow

panel = _load()
cat = category_panel(panel)
rel = relative_flow(cat, panel)
assert np.isfinite(rel["rel"].dropna()).all()

B = 1e9
m = cat.groupby("month").agg(net=("F", "sum"), gross=("F", lambda s: s.abs().sum()))
m["ratio"] = m["net"] / m["gross"]
print(f"raw flow:  mean net {m.net.mean()/B:+.2f}B/mo, total {m.net.sum()/B:+.1f}B/{len(m)}mo, "
      f"median |net|/gross {m.ratio.abs().median():.1%}, months>0 {(m.net>0).mean():.0%}")

ok = rel.dropna(subset=["rel", "aum_prev"])
w = ok.groupby("month").apply(lambda d: (d["aum_prev"] * d["rel"]).sum(), include_groups=False)
scale = ok.groupby("month").apply(lambda d: (d["aum_prev"] * d["rel"].abs()).sum(), include_groups=False)
print(f"invariant: max |sum_c A_prev*rel| = {w.abs().max():,.6f} dollars "
      f"(relative {np.nanmax(w.abs()/scale):.1e})")

u = ok.groupby("month")["rel"].sum()
print(f"unweighted sum of rel: mean |.| = {u.abs().mean():.4f}, max |.| = {u.abs().max():.4f} (pp/mo, not zero)")

raw flow:  mean net +0.57B/mo, total +46.2B/81mo, median |net|/gross 42.2%, months>0 57%
invariant: max |sum_c A_prev*rel| = 0.000001 dollars (relative 1.9e-16)
unweighted sum of rel: mean |.| = 0.0774, max |.| = 0.8588 (pp/mo, not zero)


## 2026-06-11 - Gross rotation turnover

**Problem.** The conservation identity (eq:flow_conservation) establishes that the
$A_{t-1}$-weighted relative flows cancel each month. The cancellation is used only as
a validation check; the common magnitude of the two cancelling legs is not used
anywhere, yet it has a direct reading: the number of dollars that moved between
sectors that month. We want a single monthly series measuring rotation intensity.

**Definitions** (pure panel accounting; nothing beyond eq:relative_flow and
eq:universe_g -- every step is model-free). The dollar rotation flow of category $c$:

$$R_{c,t} = A_{c,t-1}\,\mathrm{rel}_{c,t} = F_{c,t} - w_{c,t} F_{U,t}, \qquad
w_{c,t} = \frac{A_{c,t-1}}{\sum_{c'} A_{c',t-1}},\quad F_{U,t} = \sum_{c'} F_{c',t},$$

the dollars of flow into $c$ in excess of its pro-rata share of the universe tide,
giving the decomposition $F_{c,t} = w_{c,t}F_{U,t} + R_{c,t}$ (tide + rotation). By
eq:flow_conservation $\sum_c R_{c,t} = 0$, so the positive legs match the negative
legs dollar for dollar, and either side is the gross rotation turnover

$$T_t = \tfrac{1}{2}\sum_c |R_{c,t}|
      = \sum_c \max(R_{c,t}, 0) = -\sum_c \min(R_{c,t}, 0),$$

with the scale-free turnover rate $\tau_t = T_t / \sum_c A_{c,t-1}$ (the fraction of
universe AUM that rotated in month $t$). $T_t$ is the minimum total dollar mass that
must move between sectors to turn the pro-rata allocation $\{w_{c,t} F_{U,t}\}$ into
the observed allocation $\{F_{c,t}\}$.

**Self-challenge.**
- `[CHECK]` Scope: the leg equality needs every category present with prior assets.
  Months with partial coverage must be NaN, not partial sums -- a partial sum
  understates $T$ and silently breaks the equality. Empirically 3 of 81 months are
  excluded (pre-anchor months at the start of the panel).
- `[CHECK]` Raw vs smoothed: defined on raw monthly rel, where the identity is exact.
  The rotation-graph smoothing (eq:rel_smoothed) is a display transform and destroys
  exactness (the weights $A_{t-1}$ move across the window).
- `[CHECK]` Interpretation honesty: $T_t$ is net cross-sector reallocation relative
  to pro-rata -- a lower bound on investor-level movement. Round trips within the
  month and reallocation that nets out across investors are invisible.
- `[CHECK]` Degenerate cases: flows exactly pro-rata ($F_c = w_c F_U$) give $T = 0$;
  a single category gives $\mathrm{rel} \equiv 0$, $T = 0$; two categories give
  $T = |R_1| = |R_2|$.
- `[CHECK]` Normalization choice: $\tau$ uses prior universe AUM, the same weighting
  as the identity. Normalizing by gross flow $\sum_c |F_{c,t}|$ instead would measure
  the rotational share of gross flow -- a related but different question; left out to
  keep the notation budget small.


In [ ]:
import numpy as np
import pandas as pd
from viz import _load
from categories import category_panel
from rotation import relative_flow

panel = _load()
cat = relative_flow(category_panel(panel), panel)
n_cat = cat["category"].nunique()

# Scope: T_t only where every category has rel and prior assets.
ok = cat.dropna(subset=["rel", "aum_prev"])
complete = ok.groupby("month")["category"].count() == n_cat
print(f"months: {cat['month'].nunique()} total, {complete.sum()} complete")

# Production path: R = A_prev * rel from rotation.relative_flow.
ok = ok[ok["month"].isin(complete[complete].index)]
R_prod = (ok["aum_prev"] * ok["rel"]).groupby([ok["month"], ok["category"]]).sum().unstack()

# Independent path: R = F - w*F_U straight from raw pivots, no rotation.py.
F = cat.pivot(index="month", columns="category", values="F").loc[R_prod.index]
A = cat.pivot(index="month", columns="category", values="aum_prev").loc[R_prod.index]
R_ind = F.sub(A.div(A.sum(axis=1), axis=0).mul(F.sum(axis=1), axis=0))
print(f"cross-check |R_prod - R_ind|: max {np.abs(R_prod - R_ind).max().max():.6f} dollars")

# Conservation and the three equivalent turnover forms.
resid = R_prod.sum(axis=1).abs() / R_prod.abs().sum(axis=1)
T = 0.5 * R_prod.abs().sum(axis=1)
T_pos = R_prod.clip(lower=0).sum(axis=1)
T_neg = -R_prod.clip(upper=0).sum(axis=1)
print(f"conservation: max relative residual {resid.max():.2e}")
print(f"leg equality: max rel diff half-L1 vs pos {np.abs(T/T_pos - 1).max():.2e}, "
      f"vs neg {np.abs(T/T_neg - 1).max():.2e}")

# Degenerate cases: hand-computed two-sector case; pro-rata flows -> T = 0.
Rh = pd.DataFrame({"A": [100.0, 300.0], "F": [10.0, -2.0]})
Rh["R"] = Rh["F"] - Rh["A"] / Rh["A"].sum() * Rh["F"].sum()
assert abs(0.5 * Rh["R"].abs().sum() - 8.0) < 1e-12, "hand case: T must be 8"
wp = A.iloc[40] / A.iloc[40].sum()
Tp = 0.5 * (wp * 5e9 - wp * (wp * 5e9).sum()).abs().sum()
assert Tp < 1e-3, "pro-rata flows: T must be 0"
print("degenerate checks: two-sector hand case T=8 OK, pro-rata flows T=0 OK")

# The series itself.
B = 1e9
tau = T / A.sum(axis=1)
print(f"\nT_t over {len(T)} months: median {T.median()/B:.2f}B, max {T.max()/B:.2f}B")
print(f"tau_t: median {tau.median():.2%}, max {tau.max():.2%} of universe AUM/mo")
print("\ntop 5 turnover months:")
for mth, v in T.nlargest(5).items():
    print(f"  {mth.date()}  T={v/B:5.2f}B  tau={tau[mth]:.2%}")


**Result.** On the live panel (81 months, 78 complete): the production path
($A_{t-1}\cdot\mathrm{rel}$) and the independent recomputation ($F - wF_U$ from raw
pivots) agree to $10^{-6}$ dollars on billion-scale legs; the conservation residual is
$1.9\times10^{-16}$ relative; the three turnover forms agree to $3\times10^{-16}$;
both degenerate cases pass.

The series reads economically: median $T = \$2.97$B/mo (median $\tau = 1.22\%$). Top
months: Jan 2021 ($\$6.70$B, the post-vaccine value rotation), Apr 2022 ($\$5.42$B,
the rate-shock rotation), Jan 2026 ($\$5.32$B, the current late-cycle rotation),
Apr 2020 ($\$4.90$B -- the COVID peak, and the highest rate at $\tau = 4.26\%$),
Nov 2024 ($\$4.89$B, post-election). Dollar $T$ and rate $\tau$ rank differently
(universe AUM roughly tripled over the sample), which is why both are kept.

Status: `[VERIFIED]`. Consensus 2026-06-11: adopt $R_{c,t}$, $T_t$, $\tau_t$ on raw
monthly relative flow with the all-or-NaN month scope. Promoted to State 2 as
eq:rotation_dollars and eq:rotation_turnover; the paper writes the pro-rata share as
$A_{c,t-1}\,g_{U,t}$ rather than $w_{c,t} F_{U,t}$ ($w$ is already the smoothing
window of eq:rel_smoothed). Implemented as rotation.rotation_turnover (State 3),
tested (State 0), exported to the web as the rotation-intensity strip.
